# 0. Imports & initialisation Spark

In [1]:
# Import des liberairies

#----Python
import os

# ----- Spark
from pyspark.sql.session import SparkSession
import pyspark.sql.functions as sf

#----Autres-------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- --Initialisation Spark ---
spark = (
    SparkSession.builder.appName('Projet')
    .config('spark.driver.memory', '4g') # autoriser le driver d'utiliser jusqu'à 4gb
    .config('spark.executer.memory','4g')
    .config('spark.sql.shuffle.partitions','50') # par defaut c'est 200, 50 pour réduire la tension sur la machine
    .config('spark.sql.execution.arrow.pyspark.enabaled' ,'true') # pour une transition rapide entre pandas et spark df
    .getOrCreate()
)

# --- Formatage du texte ---
# character Gras
B = '\033[1m' 
BE = '\033[0m'

# 0. Constantes globales du projet

In [2]:
#Chemin vers les données sur ma machine
CHEMIN_RAW = '../data/raw'
CHEMIN_INTERIM = '../data/interim'

#-------------------------------------------------------------#
# liste de validation de la structure de la données 
regles_Validation = {
     #------ temporel ----
    'AnPrel' : r'^\d{4}$'
    ,'MoPrel': r'^\d{2}$'
    ,'DtPrel': r'^\d{2}/\d{2}/\d{4}$'
        
    #------ localisation ----
    ,'NoDep': r'^\d{2,3}$' 
    
     #------ identification station ----
    ,'CdSta': r'^\d+[A-Z]?\d*$'
    ,'NmNoSta': r'^.+$'   
    ,'CdTpSta': r'^\d+$'
    ,'LbTpSta': r'^.+$'
    
    # ---------Caractéristiques de la station
    ,'CapNomSta': r'^\d+$' 
    
    #------ contexte prélèvement ----
    ,'CdPtPrel': r'^[A-Z]\d+[a-z]?$' 
    ,'CdSup': r'^\d+$'
    ,'LbSup': r'.+'
    ,'CdFrac': r'^\d+$'
    ,'NmFrac': r'.+'

    #------ parametre mesuré ----
    ,'CdPar': r'^\d+$'
    ,'LbPar': r'.+'
    
    #------Résultat ----
    ,'RsAna': r'^\s*-?\d+([.,]\d+)?\s*$'
    ,'SbUnt': r'.+'    
}

#-------------------------------------------------------------#
# 13 codes paramètres SANDRE à conserver pour le pivot
parametres_codes = [
    # Contexte
    "1552",   # Volume moyen journalier
    "1553",   # Hauteur précipitations
    
    # Pollution organique
    "1313",   # DBO5
    "1314",   # DCO
    
    # Pollution particulaire
    "1305",   # MES
    
    # Pollution azotée (5 formes)
    "1319",   # NTK (Azote Kjeldahl)
    "1335",   # NH4 (Ammonium)
    "1339",   # NO2 (Nitrites)
    "1340",   # NO3 (Nitrates)
    "1551",   # Nglobal (Azote global)
    
    # Pollution phosphorée
    "1350",   # PT (Phosphore total)
    
    # Physico-chimie
    "1301",   # Température
    "1302",   # pH
]

#-------------------------------------------------------------#


# 1. Chargement et traitement préliminaire des données

## 1.1 donnée brutes

In [3]:
# Data frame brute

df_brt = (
    spark.read
    .option('encoding', 'UTF-8')
    .option('delimiter', ';')
    .option('header', True)
    .option('inferSchema', False)
    .option('quote', '"')
    .option('escape', '"')
    .option('multiLine', True)   
    .csv(CHEMIN_RAW + '/*.csv')
)
df_brt.printSchema()

root
 |-- Année: string (nullable = true)
 |-- Mois: string (nullable = true)
 |-- Finalité analyse: string (nullable = true)
 |-- Type_Code: string (nullable = true)
 |-- Type_Libellé: string (nullable = true)
 |-- Station_No Interne: string (nullable = true)
 |-- Station_Code SANDRE: string (nullable = true)
 |-- Station_Nom: string (nullable = true)
 |-- Departement_Numero: string (nullable = true)
 |-- Commune_Code INSEE: string (nullable = true)
 |-- Commune_Libellé: string (nullable = true)
 |-- XL93: string (nullable = true)
 |-- YL93: string (nullable = true)
 |-- Station_Capacite nominale: string (nullable = true)
 |-- Prelevement_Commentaire: string (nullable = true)
 |-- Point_Prelevement: string (nullable = true)
 |-- Prelevement_Date: string (nullable = true)
 |-- Parametre_Code SANDRE: string (nullable = true)
 |-- Parametre_Libelle: string (nullable = true)
 |-- Parametre_Type: string (nullable = true)
 |-- Support_Code: string (nullable = true)
 |-- Support_Libelle: str

In [4]:
df_brt.count()

15775181

## 1.2 Donnée colonnes selectionées

In [5]:
# les alias des colonnes

# Renomage en se basant sur la convention SANDRE (le standard dans l'eau en France) _ 2 lettres de préfixe : 
# Cd = Code, Lb = Libellé, Dt = Date, Rs = Résultat, Mn = Mnémonique, St = Statut. 
# Nm = Nom, Année = An, Mois = Mo, No/Numéro = No, symbol = Sb, Type = Tp, point = pt

# Prelevement = Prel, Station, Stat, Departement = Dept, Support = Sup, Fraction = Frac,
# Parametre = Param, Unité = Unt, Analyse = Anly, Capacite = Cap,  Nominale = Nom


In [6]:
# Selectiions des colonnes utiles
df_ColSel= df_brt.select(
    #------Temporel---------#
    sf.col('Année').alias('AnPrel')
    ,sf.col('Mois').alias('MoPrel')
    ,sf.col('Prelevement_Date').alias('DtPrel')
    
    #----Localisation-------#
    ,sf.col('Departement_Numero').alias('NoDep')
    
    #-------Identification de la station ----#
    ,sf.col('Station_Code SANDRE').alias('CdSta')
    ,sf.concat_ws('__' , sf.col('Station_Nom'), sf.col('Station_No Interne')).alias('NmNoSta')
    ,sf.col('Type_Code').alias('CdTpSta')
    ,sf.col('Type_Libellé').alias('LbTpSta')

    #----- Caractéristiques de la station ------#
    ,sf.col('Station_Capacite nominale').alias('CapNomSta')
    
    #----Contexte du Prélevement-------#
    ,sf.col('Point_Prelevement').alias('CdPtPrel')
    ,sf.col('Support_Code').alias('CdSup')
    ,sf.col('Support_Libelle').alias('LbSup')
    ,sf.col('Fraction_Code').alias('CdFrac')
    ,sf.col('Fraction_Nom').alias('NmFrac')
    
    # ---- Paramètres mesuré ----#
    ,sf.col('Parametre_Code SANDRE').alias('CdPar')
    ,sf.col('Parametre_Libelle').alias('LbPar')

    #---- Resulats et mesures ---#
    ,sf.regexp_replace('Resultat_Valeur', ',' , '.').alias('RsAna')
    ,sf.col('Resultat_Unite_Symbole').alias('SbUnt')
    
)
print(f"Total lignes : {df_ColSel.count()}")
df_ColSel.printSchema()

df_ColSel.select('AnPrel','MoPrel','DtPrel','NoDep','CdSta','NmNoSta','CdTpSta','LbTpSta','CapNomSta').show(2,truncate=False)
df_ColSel.select('CdPtPrel','CdSup','LbSup','CdFrac','NmFrac','CdPar','LbPar','RsAna','SbUnt').show(2,truncate=False)


Total lignes : 15775181
root
 |-- AnPrel: string (nullable = true)
 |-- MoPrel: string (nullable = true)
 |-- DtPrel: string (nullable = true)
 |-- NoDep: string (nullable = true)
 |-- CdSta: string (nullable = true)
 |-- NmNoSta: string (nullable = false)
 |-- CdTpSta: string (nullable = true)
 |-- LbTpSta: string (nullable = true)
 |-- CapNomSta: string (nullable = true)
 |-- CdPtPrel: string (nullable = true)
 |-- CdSup: string (nullable = true)
 |-- LbSup: string (nullable = true)
 |-- CdFrac: string (nullable = true)
 |-- NmFrac: string (nullable = true)
 |-- CdPar: string (nullable = true)
 |-- LbPar: string (nullable = true)
 |-- RsAna: string (nullable = true)
 |-- SbUnt: string (nullable = true)



+------+------+----------+-----+------------+----------------------------------+-------+-------------------------+---------+
|AnPrel|MoPrel|DtPrel    |NoDep|CdSta       |NmNoSta                           |CdTpSta|LbTpSta                  |CapNomSta|
+------+------+----------+-----+------------+----------------------------------+-------+-------------------------+---------+
|2022  |03    |08/03/2022|02   |030200501000|AGUILCOURT Le Bourg__020050101_029|29     |STEP collectivité locale |350      |
|2022  |03    |08/03/2022|02   |030200501000|AGUILCOURT Le Bourg__020050101_029|29     |STEP collectivité locale |350      |
+------+------+----------+-----+------------+----------------------------------+-------+-------------------------+---------+
only showing top 2 rows


+--------+-----+-----+------+---------+-----+---------+------+--------+
|CdPtPrel|CdSup|LbSup|CdFrac|NmFrac   |CdPar|LbPar    |RsAna |SbUnt   |
+--------+-----+-----+------+---------+-----+---------+------+--------+
|A3      |3    |Eau  |23    |Eau brute|1039 |M.O      |301.67|mg/L    |
|A3      |3    |Eau  |23    |Eau brute|1095 |DBO5 ad2 |190.00|mg(O2)/L|
+--------+-----+-----+------+---------+-----+---------+------+--------+
only showing top 2 rows


## 1.3 Validation de la structure des donnée

In [7]:
# liste de validation de la structure de la données 
regles_Validation = {
     #------ temporel ----
    'AnPrel' : r'^\d{4}$'
    ,'MoPrel': r'^\d{2}$'
    ,'DtPrel': r'^\d{2}/\d{2}/\d{4}$'
        
    #------ localisation ----
    ,'NoDep': r'^\d{2,3}$' 
    
     #------ identification station ----
    ,'CdSta': r'^\d+[A-Z]?\d*$'
    ,'NmNoSta': r'^.+$'   
    ,'CdTpSta': r'^\d+$'
    ,'LbTpSta': r'^.+$'
    
    # ---------Caractéristiques de la station
    ,'CapNomSta': r'^\d+$' 
    
    #------ contexte prélèvement ----
    ,'CdPtPrel': r'^[A-Z]\d+[a-z]?$' 
    ,'CdSup': r'^\d+$'
    ,'LbSup': r'.+'
    ,'CdFrac': r'^\d+$'
    ,'NmFrac': r'.+'

    #------ parametre mesuré ----
    ,'CdPar': r'^\d+$'
    ,'LbPar': r'.+'
    
    #------Résultat ----
    ,'RsAna': r'^\s*-?\d+([.,]\d+)?\s*$'
    ,'SbUnt': r'.+'
    
}

# Unit tests

# df_ColSel.select(sf.col('CdSta')).filter(sf.col('CdSta').rlike(r'^\d+[A-Z]\d*$')).show(5)
# df_ColSel.select(sf.col('NmNoSta')).filter(~sf.col('NmNoSta').rlike(r'^.+__\d*_\d+$')).show(5)
# df_ColSel.select(sf.col('CdTpSta')).filter(sf.col('CdTpSta').rlike(r'^\d{2}$')).show(5)

In [8]:
# Test de validation de la structure de la données
lignes_tot = df_ColSel.count()
print(f'Nombre de lignes totale: {lignes_tot}')

###----tableau de synthèse---------
#en-tete
print(f"{B} {'Colonne':<10} | {'Nb Nulls':>10} %{'Nulls':>10} | {'Nb Anomalies':>10} %{'Anomalies':>10} | {'Statute':<10} | {'Echantillon':<10} {BE}")
print(f"{B} {'--'*63} {BE}")

# Boucle unique : 1 passe par colonne pour calculer null + anomalies
for coln, pattern in regles_Validation.items():
    nulls = df_ColSel.filter((sf.col(coln).isNull())|(sf.col(coln)=="")).count()
    anomalies = df_ColSel.filter(~sf.col(coln).rlike(pattern)).count()
    pct_n = 100 * nulls/lignes_tot
    pct_a = 100 * anomalies/lignes_tot
    # Aperçu sur les anomalies
    
    valeurs_pouries = (
        df_ColSel
        .select(sf.col(coln))
        .filter(~sf.col(coln).rlike(pattern))
        .distinct()
        .limit(3)
        .collect()
    )
    if len(valeurs_pouries)==0:
        echontillon = 'aucune anomalie'
    else:
        echontillon =  [str(row[coln]) for row in valeurs_pouries]
    
    # condition d'affichage
    
    if pct_a == 0 and pct_n == 0:
        statut = 'Parfait'
    elif pct_a < 0.01 or pct_n < 0.1:
        statut = 'À Revoir'
    else:
        statut = 'Pourries'
    
    print(f" {coln:<10} | {nulls:>10} {pct_n:>10.2f}% | {anomalies:>10} {pct_a:>10.2f}% | {statut:<10} | {echontillon}\n")
    # print(f"{echontillon}")
print(f"{B} {'--'*63} {BE}") 


Nombre de lignes totale: 15775181
 Colonne    |   Nb Nulls %     Nulls | Nb Anomalies % Anomalies | Statute    | Echantillon 
 ------------------------------------------------------------------------------------------------------------------------------ 


 AnPrel     |          0       0.00% |          0       0.00% | Parfait    | aucune anomalie



 MoPrel     |          0       0.00% |          0       0.00% | Parfait    | aucune anomalie



 DtPrel     |          0       0.00% |          0       0.00% | Parfait    | aucune anomalie



 NoDep      |          0       0.00% |          0       0.00% | Parfait    | aucune anomalie



...

 NmFrac     |          0       0.00% |          0       0.00% | Parfait    | aucune anomalie



 CdPar      |          0       0.00% |          0       0.00% | Parfait    | aucune anomalie



 LbPar      |          0       0.00% |          0       0.00% | Parfait    | aucune anomalie



 RsAna      |       4983       0.03% |          0       0.00% | À Revoir   | aucune anomalie



 SbUnt      |          0       0.00% |          0       0.00% | Parfait    | aucune anomalie

 ------------------------------------------------------------------------------------------------------------------------------ 


## 1.4 donnée parametres selectionnés

In [9]:
# Aperçu des valeurs distinctes de chaque colonne

print(f"\n{B}Aperçu des toutes les valeurs distinctes par colonne{BE}\n")
print(f"{B}{'Colonne':<12} {'Count distincte':>7} {'Valeurs Distinctes':>15}{BE}")
print("-" * 65)



cols = ['AnPrel'
        ,'MoPrel'
        # , 'DtPrel'
        # ,'CdSta'
        # ,'LbStaNo'
        ,'CdTpSta'
        ,'LbTpSta'
        ,'NoDep'
        # ,'CapNomSta'
        ,'CdPtPrel'
        ,'CdSup'
        ,'LbSup'
        ,'CdFrac'
        ,'NmFrac'
        ,'CdPar'
        ,'LbPar'
        # ,'RsAna'
        ,'SbUnt'
       ]
for coln in cols:
    
    colVal_dis = (
        df_ColSel
        .select(coln)
        .distinct()
        .collect()
    )

    echantillon = [str(row[coln]) for row in colVal_dis]
    print(f"{B}{coln:<12}: {BE} {len(colVal_dis):<7} {echantillon}\n")


Aperçu des toutes les valeurs distinctes par colonne

Colonne      Count distincte Valeurs Distinctes
-----------------------------------------------------------------


AnPrel      :  5       ['2022', '2020', '2021', '2019', '2018']



MoPrel      :  12      ['03', '10', '01', '12', '05', '08', '07', '09', '06', '11', '04', '02']



CdTpSta     :  1       ['29']



LbTpSta     :  1       ['STEP collectivité locale ']



...

CdFrac      :  13      ['156', '126', '3', '130', '162', '160', '157', '22', '161', '159', '158', '23', '134']



NmFrac      :  13      ["Fraction brute de boue d'épuration", "Phase aqueuse de l'eau (filtrée, centrifugée...)", 'Phase liquide non aqueuse brute', 'Refus de degrillage brut', 'Reactifs bruts', 'Matières brutes de dessablage', "Lixiviat de boue d'épuration", 'Eau brute', "Phase particulaire de l'eau", 'Matieres brutes de curage', 'Matières brutes de vidange', "matière sèche de boue d'épuration", "Fraction inconnue de l'eau"]



CdPar       :  252     ['1095', '1307', '1782', '1374', '1436', '1372', '1394', '1241', '1383', '2831', '1449', '1355', '6484', '1295', '1440', '2835', '1305', '1116', '1245', '1392', '1304', '1393', '1823', '1822', '2833', '1337', '6455', '1060', '1041', '7009', '1433', '1242', '7037', '1552', '1099', '1420', '1551', '7040', '1386', '1799', '1115', '1244', '1314', '1239', '1395', '1448', '1243', '1096', '1339', '5440', '1317', '1553', '1350', '1246', '1347', '1335', '6018', '1439', '1821', '1098', '1367', '5946', '1319', '1379', '8727', '7039', '7431', '1382', '1388', '1356', '7075', '7007', '1303', '1313', '1434', '1385', '1047', '1447', '1841', '1048', '2840', '1371', '1302', '1340', '1781', '1375', '1191', '5931', '1039', '1824', '1311', '1301', '1362', '1387', '1369', '2521', '1042', '1389', '1312', '1451', '1338', '5479', '6104', '1000', '5932', '1332', '1106', '1443', '1444', '1933', '1667', '1082', '1474', '1506', '6366', '1621', '1170', '1955', '1140', '2910', '2915', '1206', 

LbPar       :  253     ['Volume', 'PCB 101', 'Magnésium', "Oeufs d'helminthes viable", 'Rapport Carbone sur Azote', 'Turbidité Néphélométrique', 'Plomb', "Sels d'Aluminium", 'Coliformes', 'Azote total', 'Arsenic', 'Sélénium', 'Test daphnie', 'Sulfates', 'M.O', 'Matière sèche - Ass', 'Nitrates', 'Manganèse', 'Azote global', 'Sels de Fer', 'Oxygène dissous', 'PCB 118', 'Zinc', 'Chlorures', 'Taux de saturation en O2', 'Coli. thermotolérants', 'DCO ad2', 'Matière sèche à 105°C', 'Orthophosphates', 'Mercure', 'Cadmium', 'Escherichia coli', 'Chlorophylle a', 'D.C.O.', 'Sodium', 'PCB 52', 'PCB 28', 'Micro-org. reviv. à 22°C', 'DBO5', 'Fluoranthène', 'Anhydride phosphorique', 'Nitrites', 'Température de réception ', 'Teneur en matière minéral', '7 PCBi', 'Oxyde de magnesium', 'Sulfures', 'Indice Phénol', 'Chrome', 'Fer', 'Indice hydrocarbure', 'Conductivité', 'Benzo(a)pyrène', 'PCB 180', 'Micro-Org. reviv. à 37° C', 'Chrome VI', 'Conductivité à 20°C', 'Débit instantané', 'Matières en suspensio

SbUnt       :  102     ['kg/(t MB)', 'mg(Se)/kg', '°f', 'mm', 'g/(kg MS)', '%', 'mg(P)/kg', 'µg(As)/L', 'mg(Cr)/kg', 'NFU', 'mg(SO4)/L', 'µg(Pb)/L', 'mg(P)/L', 'mg/kg', '°C', 'g(MgO)/(kg MS)', 'mg(NH4)/L', 'mg(NO3)/L', '% MS', 'mg(S)/L', 'g(CaO)/(kg MS)', 'mg(PO4)/L', 'mg(Zn)/kg', 'X', 'kg(CaO)/t', 'kg(P2O5)/t', 'mg(K)/kg', 'mg(Mo)/kg', 'kW.h', '% MB', 'n/mL', 'kg(K2O)/t', 'unité pH', 'µS/cm', 'mg(O2)/L', 'µg(Ni)/L', 'g(N)/kg', 'kg(MgO)/t', 'Unité inconnue', 'm3/s', '% poids sec', 'µg/(kg MS)', 'µg(Cr)/L', 'µg(Fe)/L', 'mg(Na)/kg', 'mg(C)/L', 'g/L', 'g/kg', 'mg/(kg MS)', 'NPP/100mL', 'µg(Zn)/L', 'min', 'm3', 'n/L', 'm3/j', 'kg(N)/t', 'µg(Cd)/L', 'µg(Hg)/L', 'kg', 'n/(5L)', 'mg(Pt)/L', 'mg(NO2)/L', 'µg(Cu)/L', 'n/(100mL)', 'mg/L', 'kg/(t MS)', 'mg(B)/kg', 'µg(B)/L', 'µg/L', 'mg(N)/L', 'mg(Cl)/L', 'n/g', 'MV', 'Equitox/m3', 'µg(Cl)/L', 'T', 'µg(Sn)/L', 'cm', 'mg(P2O5)/l ', '% fraction solide', 'n/(20mL)', 'µg(Ti)/L', 'µg(Co)/L', 'mg(Ag)/kg', 'µg/kg', 'mg(Cl2)/L', 'n/(10g)', 'NTU', 'NPP/(1

In [10]:
# Conclusion:
# CdPtPrel: filtrage sur A3 (entrée station) + A4 (sortie station) uniquement → points réglementaires obligatoires SANDRE
# , A3 = features du modèle, A4 = calcul du label de conformité.
# Points exclus: A2/A5/A6/A7/A8 (by-pass, boues, REUT hors scope),
# S* (sous-points logiques redondants avec A3/A4), M* (milieu récepteur).

In [11]:
# 13 codes paramètres SANDRE à conserver pour le pivot
parametres_codes = [
    # Contexte
    "1552",   # Volume moyen journalier
    "1553",   # Hauteur précipitations
    
    # Pollution organique
    "1313",   # DBO5
    "1314",   # DCO
    
    # Pollution particulaire
    "1305",   # MES
    
    # Pollution azotée (5 formes)
    "1319",   # NTK (Azote Kjeldahl)
    "1335",   # NH4 (Ammonium)
    "1339",   # NO2 (Nitrites)
    "1340",   # NO3 (Nitrates)
    "1551",   # Nglobal (Azote global)
    
    # Pollution phosphorée
    "1350",   # PT (Phosphore total)
    
    # Physico-chimie
    "1301",   # Température
    "1302",   # pH
]

In [12]:
df_Filt = (
    df_ColSel
    .filter(sf.col("CdPar").isin(parametres_codes))      # 13 paramètres
    .filter(sf.col("CdPtPrel").isin("A3", "A4"))         # entrée + sortie uniquement
    .filter(sf.col("CdSup") == "3")                  # eau uniquement (pas boue)
)

# Diagnostic
print(f"Avant filtrage : {df_ColSel.count():,} lignes")
print(f"Après filtrage : {df_Filt.count():,} lignes")
print(f"Ratio conservé : {100 * df_Filt.count() / df_ColSel.count():.1f}%")

# Répartition finale
df_Filt.groupBy("CdPtPrel", "LbPar", "CdPar").count() \
       .orderBy("CdPtPrel", sf.desc("count")) \
       .show(50, truncate=False)

Avant filtrage : 15,775,181 lignes


Après filtrage : 7,630,169 lignes


Ratio conservé : 48.4%


+--------+-----------------------+-----+-------+
|CdPtPrel|LbPar                  |CdPar|count  |
+--------+-----------------------+-----+-------+
|A3      |Volume moyen journalier|1552 |1965651|
|A3      |Hauteur précipitations |1553 |1705825|
|A3      |D.C.O.                 |1314 |118525 |
|A3      |Matières en suspension |1305 |118357 |
|A3      |pH                     |1302 |113332 |
|A3      |DBO5                   |1313 |96430  |
|A3      |Phosphore total        |1350 |79727  |
|A3      |Azote Kjeldahl         |1319 |79177  |
|A3      |Ammonium               |1335 |78738  |
|A3      |Azote global           |1551 |78655  |
|A3      |Nitrates               |1340 |74646  |
|A3      |Nitrites               |1339 |74608  |
|A3      |Température de l'Eau   |1301 |21569  |
|A4      |Volume moyen journalier|1552 |1949089|
|A4      |Température de l'Eau   |1301 |136034 |
|A4      |D.C.O.                 |1314 |120158 |
|A4      |Matières en suspension |1305 |119900 |
|A4      |pH        

In [13]:
# Aperçu des valeurs distinctes de chaque colonne

print(f"\n{B}Aperçu des toutes les valeurs distinctes par colonne{BE}\n")
print(f"{B}{'Colonne':<12} {'Count distincte':>7} {'Valeurs Distinctes':>15}{BE}")
print("-" * 65)



cols = ['AnPrel'
        ,'MoPrel'
        # , 'DtPrel'
        # ,'CdSta'
        # ,'LbStaNo'
        ,'CdTpSta'
        ,'LbTpSta'
        ,'NoDep'
        # ,'CapNomSta'
        ,'CdPtPrel'
        ,'CdSup'
        ,'LbSup'
        ,'CdFrac'
        ,'NmFrac'
        ,'CdPar'
        ,'LbPar'
        # ,'RsAna'
        ,'SbUnt'
       ]
for coln in cols:
    
    colVal_dis = (
        df_Filt
        .select(coln)
        .distinct()
        .collect()
    )

    echantillon = [str(row[coln]) for row in colVal_dis]
    print(f"{B}{coln:<12}: {BE} {len(colVal_dis):<7} {echantillon}\n")


Aperçu des toutes les valeurs distinctes par colonne

Colonne      Count distincte Valeurs Distinctes
-----------------------------------------------------------------


AnPrel      :  5       ['2022', '2020', '2021', '2019', '2018']



MoPrel      :  12      ['03', '10', '01', '12', '05', '08', '07', '09', '06', '11', '04', '02']



CdTpSta     :  1       ['29']



LbTpSta     :  1       ['STEP collectivité locale ']



...

CdFrac      :  4       ['3', '22', '23', '156']



NmFrac      :  4       ["Phase aqueuse de l'eau (filtrée, centrifugée...)", 'Eau brute', "Fraction inconnue de l'eau", "Phase particulaire de l'eau"]



CdPar       :  13      ['1305', '1552', '1551', '1314', '1339', '1553', '1350', '1335', '1319', '1313', '1302', '1340', '1301']



LbPar       :  13      ['Nitrates', 'Azote global', 'D.C.O.', 'DBO5', 'Nitrites', 'Matières en suspension', 'Azote Kjeldahl', 'Volume moyen journalier', 'Ammonium', "Température de l'Eau", 'Phosphore total', 'pH', 'Hauteur précipitations']



SbUnt       :  13      ['mm', 'mg(P)/kg', 'mg(P)/L', '°C', 'mg(NH4)/L', 'mg(NO3)/L', 'unité pH', 'mg(O2)/L', 'm3/j', 'mg(Pt)/L', 'mg(NO2)/L', 'mg/L', 'mg(N)/L']



In [14]:
# Sauvegarde de df_Filt
df_Filt.write.mode("overwrite").parquet(CHEMIN_INTERIM + '/df_Filt.parquet')
print("df_Filt sauvegardé en Parquet")

df_Filt sauvegardé en Parquet


In [15]:
spark.stop()